In [1]:
# Libraries to import:
import pandas as pd
import datetime
import webbrowser
import math
import folium
from haversine import haversine, Unit

# Custom class which stores address points:
class MapPoints:

    def __init__ (self, zipcode, coordinates, maxdistance_km):
        self.zipcode = zipcode
        self.coordinates = coordinates
        self.maxdistance_km = maxdistance_km

# This function converts the zip code from float to integer:
def ConvertZipCodeToInteger(row):
    val = 0
    if pd.notnull(row['ZipCode']):
        #val = row['ZipCode'].astype(int)
        val = int(row['ZipCode'])
        
    return val

# This function merged all the address fields:
def CalcFullAddress(row):

    FullAddress = ""

    if pd.notnull(row['PrefixAddressNumber']):
        FullAddress = FullAddress + row['PrefixAddressNumber'] + " "

    if pd.notnull(row['AddressNumber']):
        FullAddress = FullAddress + str(row['AddressNumber']) + " "

    if pd.notnull(row['SuffixAddressNumber']):
        FullAddress = FullAddress + row['SuffixAddressNumber'] + " "
        
    if pd.notnull(row['CompleteStreetName']):
        FullAddress = FullAddress + row['CompleteStreetName'] + ", "

    if pd.notnull(row['CityTownName']):
        FullAddress = FullAddress + row['CityTownName'] + ", "

    if pd.notnull(row['State']):
        FullAddress = FullAddress + row['State'] + ", "

    if pd.notnull(row['ZipCodeInteger']):
        FullAddress = FullAddress + str(row['ZipCodeInteger']) + ", United States"
    
    return FullAddress

# Calculates distance between 2 coordinates:
def haversine(coord1, coord2):
    # Radius of the Earth in kilometers
    R = 6371.0
    
    # Coordinates in decimal degrees
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in kilometers
    distance = R * c
    
    return distance

# Main processing:

# Import zipcode geocoding file into data frame:
print (datetime.datetime.now())
print ("Importing the zip code geocoding file...")
df1 = pd.read_csv("zip_lat_long.csv", delimiter=',')

# Convert zipcode to integer, so we can join on zipcode, to get the zipcode coordinates:
df1['ZIP'] = df1['ZIP'].astype(int)

# Import Long Island zip codes. We will use these to filter the NY data.
dfLongIslandZipcode = pd.read_csv("ZipCodesLongIslandOnly.txt")

# Import NY addresses into data frame:
print (datetime.datetime.now())
print ("Importing the NY addresses...")
df2 = pd.read_csv("NYS_Address_Points_2452783919555667456.csv", delimiter = ",")

# Convert zipcode to integer, so we can join on zipcode, to get the zipcode coordinates:
print (datetime.datetime.now())
print ("Converting zipcodes to integers...")
df2['ZipCodeInteger'] = df2.apply(ConvertZipCodeToInteger, axis=1)

# Merge address fields:
#print (datetime.datetime.now())
#print ("Merging address fields into a full address field...")
#df2['FullAddress'] = df2.apply(CalcFullAddress,axis=1)

# Copy to new dataset and delete current dataframe to save space:
#df3 = df2[ ['FullAddress','ZipCodeInteger'] ]
df3 = df2[ ['ZipCodeInteger'] ]
del df2

# Geocode NY addresses by merging both data frames:
print (datetime.datetime.now())
print ("Geocoding zipcodes...")
df3 = df3.merge(df1,left_on='ZipCodeInteger', right_on='ZIP')

# Remove duplicates:
df3.drop_duplicates(inplace=True)

# Load all addresses into custom class list:
AllMapPoints = []
for index, row in df3.iterrows():
    
    if pd.notnull(row['ZipCodeInteger']) and pd.notnull(row['LAT']) and pd.notnull(row['LNG']):
        # Create one instance of custom class:
        OneMapPoint = MapPoints (row['ZipCodeInteger'], (row['LAT'], row['LNG']), None )

        # Append instance to list of all map points"
        AllMapPoints.append (OneMapPoint)

print ("Total # of map points...")
print (len(AllMapPoints))

# Calculate distance between points:
print (datetime.datetime.now())
print ("Calculating distance between points...")

for Point1 in AllMapPoints:

    #print ("--------------------------------")
    #print (Point1.zipcode)

    MaxDistance = 0
    
    for Point2 in AllMapPoints:
    
        if Point1.coordinates != None and Point2.coordinates != None:
        
            distance = haversine (Point1.coordinates, Point2.coordinates)

            #print (distance)

            if distance > 0 and distance > MaxDistance:
                MaxDistance = distance
    
    #print ("Maximum distance..,")
    #print (MaxDistance)
    Point1.maxdistance_km = MaxDistance

# Define map:
map = folium.Map(location = [42.804977, -75.261071], zoom_start = 7)

# Add all coordinates to map::
for OnePoint in AllMapPoints:
    if OnePoint.coordinates != None:
            folium.Marker (location=[OnePoint.coordinates[0],OnePoint.coordinates[1]], popup=OnePoint.coordinates, icon=folium.Icon(color="red")).add_to(map)

# Calculating central point:
CentralPoint_Zipcode = ""
CentralPoint_Coordinates = ()
CentralPoint_Distance = 999999

for point in AllMapPoints:
    
    if point.maxdistance_km > 0 and  point.maxdistance_km < CentralPoint_Distance:
        
        CentralPoint_Zipcode = point.zipcode
        CentralPoint_Coordinates = point.coordinates
        CentralPoint_Distance = point.maxdistance_km

print ("Central Point...")
print (CentralPoint_Zipcode)
print (CentralPoint_Coordinates)
print (CentralPoint_Distance)

#Add central point to map:
folium.Marker (location=[CentralPoint_Coordinates[0],CentralPoint_Coordinates[1]], popup="Central Point", icon=folium.Icon(color="green")).add_to(map)

# Save the map:
map.save("NewYorkMap.html")
webbrowser.open_new_tab("NewYorkMap.html")

print ("Completed")
print (datetime.datetime.now())


2025-05-26 14:45:48.183799
Importing the zip code geocoding file...
2025-05-26 14:45:48.205091
Importing the NY addresses...


C:\Users\ljokh\AppData\Local\Temp\ipykernel_17068\3639305326.py:93: DtypeWarning: Columns (0,2,3,4,5,6,7,8,9,10,13,14,15,16,17,18,19,20,21,22,28,33,35,39,41,43,44,45,46,47,48,49,50,53,54,55,56,57,58,59) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("NYS_Address_Points_2452783919555667456.csv", delimiter = ",")


2025-05-26 14:48:00.463717
Converting zipcodes to integers...
2025-05-26 14:51:56.677933
Geocoding zipcodes...
Total # of map points...
1684
2025-05-26 14:51:58.927731
Calculating distance between points...
Central Point...
13830.0
(np.float64(42.437694), np.float64(-75.627357))
342.24718349254755
Completed
2025-05-26 14:52:18.617585
